# Notebook 03: Análisis, Reglas de Fraude & Reportes

## Objetivos
- Lectura de la capa Gold (Delta Lake)
- Reglas basadas en umbrales y z-scores
- Spark SQL: consultas complejas de análisis
- Export de resultados para dashboard
- Time travel: comparación temporal
- Z-ordering y OPTIMIZE

In [1]:
import sys
sys.path.insert(0, '/home/jovyan')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

from src.config import SILVER_PATH, GOLD_PATH, SPARK_APP_NAME
from src.utils import benchmark, setup_logging

setup_logging()

In [2]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}-analysis")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.adaptive.enabled", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## 1. Lectura de Datos

In [3]:
# Leer Silver para análisis detallado
silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_df.createOrReplaceTempView("transactions")

# Leer Gold para métricas agregadas
gold_df = spark.read.format("delta").load(GOLD_PATH)
gold_df.createOrReplaceTempView("fraud_metrics")

print(f"Silver: {silver_df.count():,} registros")
print(f"Gold: {gold_df.count():,} registros")

Silver: 100,000 registros
Gold: 90 registros


## 2. Análisis con Spark SQL

### 2a. Top 10 cuentas con mayor monto de fraude

In [4]:
spark.sql("""
    SELECT 
        account_id,
        COUNT(*) as total_txns,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_txns,
        ROUND(SUM(CASE WHEN is_fraud = 1 THEN amount ELSE 0 END), 2) as fraud_amount,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(AVG(risk_score), 4) as avg_risk_score
    FROM transactions
    GROUP BY account_id
    HAVING fraud_txns > 0
    ORDER BY fraud_amount DESC
    LIMIT 10
""").show(truncate=False)

+------------+----------+----------+------------+----------+--------------+
|account_id  |total_txns|fraud_txns|fraud_amount|avg_amount|avg_risk_score|
+------------+----------+----------+------------+----------+--------------+
|ACC_00064962|1         |1         |517.82      |517.82    |0.8147        |
|ACC_00075113|1         |1         |517.69      |517.69    |0.9102        |
|ACC_00070020|1         |1         |515.87      |515.87    |0.7143        |
|ACC_00038066|1         |1         |515.03      |515.03    |0.9595        |
|ACC_00045975|1         |1         |511.39      |511.39    |0.9177        |
|ACC_00007477|1         |1         |510.37      |510.37    |0.7079        |
|ACC_00069714|1         |1         |507.75      |507.75    |0.9914        |
|ACC_00095697|1         |1         |507.5       |507.5     |0.7108        |
|ACC_00066927|1         |1         |507.47      |507.47    |0.9844        |
|ACC_00054516|1         |1         |504.88      |504.88    |0.7845        |
+-----------

### 2b. Distribución horaria de fraude

In [5]:
hourly_fraud = spark.sql("""
    SELECT 
        hour,
        COUNT(*) as total_txns,
        SUM(is_fraud) as fraud_txns,
        ROUND(SUM(is_fraud) / COUNT(*) * 100, 2) as fraud_rate_pct
    FROM transactions
    GROUP BY hour
    ORDER BY hour
""")

hourly_fraud.show(24)

+----+----------+----------+--------------+
|hour|total_txns|fraud_txns|fraud_rate_pct|
+----+----------+----------+--------------+
|   0|      4237|        43|          1.01|
|   1|      4132|        58|           1.4|
|   2|      4207|        53|          1.26|
|   3|      4319|        43|           1.0|
|   4|      4220|        62|          1.47|
|   5|      4232|        56|          1.32|
|   6|      4270|        59|          1.38|
|   7|      4143|        54|           1.3|
|   8|      4204|        46|          1.09|
|   9|      4085|        56|          1.37|
|  10|      4207|        48|          1.14|
|  11|      4161|        57|          1.37|
|  12|      4117|        45|          1.09|
|  13|      4128|        37|           0.9|
|  14|      4152|        45|          1.08|
|  15|      4162|        57|          1.37|
|  16|      4178|        43|          1.03|
|  17|      4090|        50|          1.22|
|  18|      4101|        55|          1.34|
|  19|      4126|        40|    

### 2c. Categorías de merchant con mayor riesgo

In [6]:
spark.sql("""
    SELECT 
        merchant_category,
        total_transactions,
        total_fraud,
        ROUND(fraud_rate * 100, 4) as fraud_rate_pct,
        avg_fraud_score,
        fraud_amount,
        high_risk_count
    FROM fraud_metrics
    ORDER BY fraud_rate DESC
    LIMIT 15
""").show(truncate=False)

+-----------------+------------------+-----------+--------------+---------------+------------+---------------+
|merchant_category|total_transactions|total_fraud|fraud_rate_pct|avg_fraud_score|fraud_amount|high_risk_count|
+-----------------+------------------+-----------+--------------+---------------+------------+---------------+
|insurance        |1010              |26         |2.5743        |0.0451         |2297.58     |0              |
|home_improvement |1042              |23         |2.2073        |0.0413         |3213.17     |0              |
|online_shopping  |1019              |22         |2.159         |0.0463         |2563.82     |0              |
|grocery          |1083              |21         |1.9391        |0.048          |3144.61     |0              |
|travel           |1187              |21         |1.7692        |0.0458         |1943.09     |0              |
|subscription     |1135              |20         |1.7621        |0.0433         |3149.51     |0              |
|

### 2d. Análisis de transacciones por canal y fin de semana

In [7]:
spark.sql("""
    SELECT 
        channel,
        is_weekend,
        COUNT(*) as total_txns,
        SUM(is_fraud) as fraud_txns,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(SUM(is_fraud) / COUNT(*) * 100, 4) as fraud_rate_pct
    FROM transactions
    GROUP BY channel, is_weekend
    ORDER BY fraud_rate_pct DESC
""").show(truncate=False)

+-------+----------+----------+----------+----------+--------------+
|channel|is_weekend|total_txns|fraud_txns|avg_amount|fraud_rate_pct|
+-------+----------+----------+----------+----------+--------------+
|pos    |1         |5725      |83        |37.48     |1.4498        |
|branch |1         |5623      |79        |38.31     |1.4049        |
|mobile |1         |5708      |77        |37.55     |1.349         |
|atm    |1         |5754      |73        |37.69     |1.2687        |
|mobile |0         |14294     |177       |37.75     |1.2383        |
|atm    |0         |14403     |175       |37.51     |1.215         |
|branch |0         |14320     |169       |37.83     |1.1802        |
|online |0         |14326     |165       |37.56     |1.1518        |
|online |1         |5514      |59        |36.97     |1.07          |
|pos    |0         |14333     |153       |37.6      |1.0675        |
+-------+----------+----------+----------+----------+--------------+



## 3. Delta Lake: Time Travel

In [8]:
# Ver historial de versiones de la tabla Gold
delta_table = DeltaTable.forPath(spark, GOLD_PATH)
history = delta_table.history()
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-----------------------+---------+------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                            |
+-------+-----------------------+---------+------------------------------------------------------------+
|0      |2026-03-26 23:30:13.843|WRITE    |{numFiles -> 1, numOutputRows -> 90, numOutputBytes -> 5796}|
+-------+-----------------------+---------+------------------------------------------------------------+



In [9]:
# Leer una versión anterior (si existe)
try:
    gold_v0 = spark.read.format("delta").option("versionAsOf", 0).load(GOLD_PATH)
    print(f"Versión 0: {gold_v0.count()} registros")
    
    gold_latest = spark.read.format("delta").load(GOLD_PATH)
    print(f"Versión actual: {gold_latest.count()} registros")
except Exception as e:
    print(f"Solo hay una versión disponible: {e}")

Versión 0: 90 registros
Versión actual: 90 registros


## 4. Delta Lake: OPTIMIZE y Z-Ordering

In [10]:
# OPTIMIZE + Z-ORDER en Silver por account_id (mejora consultas por cuenta)
with benchmark("OPTIMIZE + Z-ORDER Silver por account_id"):
    silver_delta = DeltaTable.forPath(spark, SILVER_PATH)
    silver_delta.optimize().executeZOrderBy("account_id")

print("✅ Z-Ordering aplicado")

2026-03-26 23:30:53 | src.utils | INFO | ⏱ Iniciando: OPTIMIZE + Z-ORDER Silver por account_id
2026-03-26 23:31:01 | src.utils | INFO | ✅ OPTIMIZE + Z-ORDER Silver por account_id completado en 8.29 segundos


✅ Z-Ordering aplicado


In [11]:
# Benchmark: lectura con y sin Z-ordering
test_account = "ACC_00000001"

with benchmark(f"Consulta filtrada por account_id (post Z-order)"):
    result = (
        spark.read.format("delta").load(SILVER_PATH)
        .where(F.col("account_id") == test_account)
        .count()
    )
    print(f"Transacciones de {test_account}: {result:,}")

2026-03-26 23:31:01 | src.utils | INFO | ⏱ Iniciando: Consulta filtrada por account_id (post Z-order)
2026-03-26 23:31:04 | src.utils | INFO | ✅ Consulta filtrada por account_id (post Z-order) completado en 2.37 segundos


Transacciones de ACC_00000001: 1


## 5. Export para Dashboard

In [12]:
# Los datos Gold ya están en Delta Lake, Streamlit los lee via deltalake (Python)
print(f"\n{'='*60}")
print("📊 Datos disponibles para el dashboard:")
print(f"  Gold Layer: {GOLD_PATH}")
print(f"  Registros: {gold_df.count():,}")
print(f"{'='*60}")

print("\n✅ Notebook 03 completado. Dashboard listo en http://localhost:8501")


📊 Datos disponibles para el dashboard:
  Gold Layer: /home/jovyan/data/delta/gold/fraud_metrics
  Registros: 90

✅ Notebook 03 completado. Dashboard listo en http://localhost:8501
